# Coordinate-Based PINN for Tumor Growth Prediction

Proper Physics-Informed Neural Network following Zhang et al. (2025):
- MLP takes (x, y, z, t, treatment) coordinates as input
- Outputs tumor density u at each point
- Exact PDE derivatives via torch.autograd.grad
- Learns diffusion (D) and proliferation (rho) per transition
- Fisher-KPP: du/dt = D*laplacian(u) + rho*u*(1-u)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the repo and install
!rm -rf /content/BINNs
!git clone https://github.com/tanushappapogu-max/BINNs.git /content/BINNs
%cd /content/BINNs
!pip install -e '.[ml,imaging]' -q

In [ ]:
# Verify GPU availability
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Configure paths
from pathlib import Path

DRIVE_DATA = Path('/content/drive/MyDrive/STS_Project_Data/data')
DERIVED = DRIVE_DATA / 'derived' / 'mu_glioma_post'

TRAIN_INDEX = DERIVED / 'shared_training_transitions.json'
VAL_INDEX = DERIVED / 'shared_model_selection_transitions.json'
MANIFEST = DERIVED / 'shared_split_manifest.json'
DATA_ROOT = Path('/content/drive/MyDrive/STS_Project_Data')
OUTPUT_ROOT = DRIVE_DATA / 'outputs' / 'unet_pinn'

for p in [TRAIN_INDEX, VAL_INDEX, MANIFEST]:
    assert p.exists(), f'Missing: {p}'
print('All paths verified')

In [ ]:
from gbm_pinn.unet_forecaster import TrainConfig, train

config = TrainConfig(
    transition_index_path=TRAIN_INDEX,
    manifest_path=MANIFEST,
    val_transition_index_path=VAL_INDEX,
    data_root=DATA_ROOT,
    output_root=OUTPUT_ROOT,
    device='cuda',
    downsample=2,
    hidden_dim=256,
    hidden_layers=4,
    n_data_points=8192,
    n_collocation_points=4096,
    pinn_epochs=500,
    pinn_lr=1e-3,
    data_weight=10.0,
    physics_weight=1.0,
)

results = train(config)

In [ ]:
# Results summary
import json
results = json.load(open(OUTPUT_ROOT / 'results.json'))
print(f"Mean Dice: {results['mean_dice']:.4f}")
print(f"Mean Skill: {results['mean_skill']:+.4f}")
print(f"Beating Persistence: {results['n_beating_persistence']}/{results['n_total']}")
print(f"\nPer-transition results:")
for r in results.get('records', []):
    if 'forecast_dice' not in r:
        print(f"  {r['transition_id']}: FAILED - {r.get('error', 'unknown')}")
        continue
    skill = r['dice_skill']
    marker = '+' if skill > 0 else ''
    print(f"  {r['transition_id']}: Dice={r['forecast_dice']:.4f} skill={marker}{skill:.4f} D={r['learned_D']:.4f} rho={r['learned_rho']:.5f}")